# 01 - Fine-tune a dish classifier (RQ1)

Goal: recognise dishes from photos. Pretrained EfficientNet-B0 is fine-tuned on a few Food-101 classes plus our own Vietnamese dish photos.
Run on Google Colab: Runtime > Change runtime type > GPU. Sources: Food-101 (Bossard et al., 2014), EfficientNet (Tan & Le, 2019).

Own photos live on Google Drive: `MyDrive/own_photos/<dish_name>/*.jpg` (one folder per dish; every folder becomes a class).

In [ ]:
!pip -q install onnx onnxruntime onnxscript scikit-learn

Mount Google Drive first: it holds our photos and keeps checkpoints if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. Config
Vietnamese classes are read from the folder names in `own_photos`. `EPOCHS` is the only value to change for a quick trial run (set 1).

In [ ]:
import hashlib, json, os, random, time
from pathlib import Path

import numpy as np
import torch, torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms

FOOD_CLASSES = ["spring_rolls", "fried_rice", "ramen", "sushi", "pizza",
                "hamburger", "french_fries", "caesar_salad", "omelette"]   # Food-101 folder names
OWN_DIR = Path("/content/drive/MyDrive/own_photos")    # our Vietnamese photos
ROOT = Path("/content/data")                           # Food-101 download (~5 GB, stays on Colab)
OUT = Path("/content/drive/MyDrive/food_ai")           # checkpoints and exported model
OUT.mkdir(parents=True, exist_ok=True)

VN_CLASSES = sorted(d.name for d in OWN_DIR.iterdir() if d.is_dir())
CLASSES = FOOD_CLASSES + VN_CLASSES
print("classes:", CLASSES)

EPOCHS = 5   # set to 1 for a quick trial run
BATCH, LR, SEED = 32, 1e-3, 42
MAX_FOOD_PER_CLASS = 200     # cap Food-101 train images per class so Vietnamese classes are not drowned
VAL_FRAC, TEST_FRAC = 0.2, 0.2   # share of own photos per class held out for val / test

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 2. Data
Food-101 has an official train/test split; 10% of its train part becomes validation. Own photos are split per class into train/val/test, after dropping byte-identical duplicates (common when images are downloaded from the web), so one image can never sit in two splits.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}

def split_own(root, names, idx, val_frac, test_frac, seed):
    # Per-class split of own photos. Exact duplicates are dropped first.
    rng, seen, dropped = random.Random(seed), set(), 0
    out = {"train": [], "val": [], "test": []}
    for c in names:
        uniq = []
        for p in sorted(q for q in (root / c).iterdir() if q.suffix.lower() in IMG_EXT):
            h = hashlib.md5(p.read_bytes()).hexdigest()
            if h in seen: dropped += 1; continue
            seen.add(h); uniq.append(p)
        assert len(uniq) >= 5, f"{c}: need at least 5 distinct images, found {len(uniq)}"
        rng.shuffle(uniq)
        nt, nv = max(1, round(len(uniq) * test_frac)), max(1, round(len(uniq) * val_frac))
        for split, part in (("test", uniq[:nt]), ("val", uniq[nt:nt + nv]), ("train", uniq[nt + nv:])):
            out[split] += [(str(p), idx[c]) for p in part]
    return out, dropped

In [ ]:
food = {s: datasets.Food101(ROOT, split=s, download=True) for s in ("train", "test")}
cls_idx = {c: i for i, c in enumerate(CLASSES)}
food_to_new = {food["train"].class_to_idx[c]: cls_idx[c] for c in FOOD_CLASSES}

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_train = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
                               transforms.RandomHorizontalFlip(), transforms.ToTensor(), norm])
tf_eval = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                              transforms.ToTensor(), norm])

class Items(Dataset):
    # items: list of (image_path, label)
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert("RGB")), y

def food_items(split):
    ds = food[split]
    return [(str(ds._image_files[i]), food_to_new[l]) for i, l in enumerate(ds._labels) if l in food_to_new]

def cap(items, n):
    seen, out = {}, []
    for p, y in items:
        if seen.get(y, 0) < n: out.append((p, y)); seen[y] = seen.get(y, 0) + 1
    return out

train_all = food_items("train")
random.shuffle(train_all)
n_val = len(train_all) // 10
own, dropped = split_own(OWN_DIR, VN_CLASSES, cls_idx, VAL_FRAC, TEST_FRAC, SEED)
train_items = cap(train_all[n_val:], MAX_FOOD_PER_CLASS) + own["train"]
val_items = train_all[:n_val] + own["val"]
test_items = food_items("test") + own["test"]

# leakage check: no image path appears in two splits
paths = [set(p for p, _ in s) for s in (train_items, val_items, test_items)]
assert not (paths[0] & paths[1] or paths[0] & paths[2] or paths[1] & paths[2]), "split leakage"

print("duplicate own photos dropped:", dropped)
for name, items in (("train", train_items), ("val", val_items), ("test", test_items)):
    print(f"{name:5s} {len(items):5d}", dict(zip(CLASSES, np.bincount([y for _, y in items], minlength=len(CLASSES)))))

# rare classes get a larger loss weight so the few Vietnamese photos still count; square root keeps it
# mild (full inverse-frequency weights made the model predict Vietnamese dishes for foreign food)
counts = np.bincount([y for _, y in train_items], minlength=len(CLASSES))
class_w = torch.tensor((counts.sum() / (len(CLASSES) * counts)) ** 0.5, dtype=torch.float32)

loaders = {name: DataLoader(Items(items, tf), batch_size=BATCH, shuffle=(name == "train"), num_workers=2)
           for name, items, tf in (("train", train_items, tf_train), ("val", val_items, tf_eval),
                                   ("test", test_items, tf_eval))}

## 3. Model
EfficientNet-B0 pretrained on ImageNet: small enough for free Colab and CPU inference. The classifier head is trained first (fast, keeps pretrained features intact), then the whole network is fine-tuned with a lower learning rate.

In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASSES))
model.to(DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=class_w.to(DEVICE), label_smoothing=0.1)

def run_epoch(loader, opt=None):
    model.train(opt is not None)
    tot, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(opt is not None):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x); loss = loss_fn(out, y)
            if opt: opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); n += len(y)
    return tot / n, correct / n

def fit(params, lr, epochs, tag):
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    best = 0
    for e in range(epochs):
        t = time.time()
        tl, ta = run_epoch(loaders["train"], opt)
        vl, va = run_epoch(loaders["val"])
        print(f"[{tag}] epoch {e+1}/{epochs} train {tl:.3f}/{ta:.3f}  val {vl:.3f}/{va:.3f}  {time.time()-t:.0f}s")
        if va > best:
            best = va; torch.save(model.state_dict(), OUT / "best.pt")   # checkpoint on best val
    return best

for p in model.features.parameters(): p.requires_grad = False
fit(model.classifier.parameters(), LR, 2, "head")
for p in model.parameters(): p.requires_grad = True
best_val = fit(model.parameters(), LR / 10, EPOCHS, "full")
print("best val accuracy:", round(best_val, 4))

## 4. Evaluation (answers RQ1)
Evaluated once on the untouched test split. Vietnamese classes are reported separately: with only a handful of test photos per dish those numbers are rough, so report the photo count next to them.

In [ ]:
model.load_state_dict(torch.load(OUT / "best.pt", map_location=DEVICE)); model.eval()
ys, ps = [], []
with torch.no_grad():
    for x, y in loaders["test"]:
        ps += model(x.to(DEVICE)).argmax(1).cpu().tolist(); ys += y.tolist()
ys, ps = np.array(ys), np.array(ps)
labels = list(range(len(CLASSES)))
print("test accuracy:", round(accuracy_score(ys, ps), 4), " macro F1:", round(f1_score(ys, ps, average="macro"), 4))
vn = ys >= len(FOOD_CLASSES)
print(f"Food-101 classes accuracy: {(ys[~vn] == ps[~vn]).mean():.4f} ({(~vn).sum()} test images)")
print(f"Vietnamese classes accuracy: {(ys[vn] == ps[vn]).mean():.4f} ({vn.sum()} test images)")
print(classification_report(ys, ps, labels=labels, target_names=CLASSES, digits=3, zero_division=0))

import matplotlib.pyplot as plt
cm = confusion_matrix(ys, ps, labels=labels)
print(cm)
plt.figure(figsize=(9, 8)); plt.imshow(cm, cmap="Blues")
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        if cm[i, j]: plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                              color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xticks(labels, CLASSES, rotation=90); plt.yticks(labels, CLASSES)
plt.xlabel("predicted"); plt.ylabel("true"); plt.colorbar(); plt.tight_layout()
plt.savefig(OUT / "confusion_matrix.png", dpi=150); plt.show()

## 5. Export for the web app
Exports ONNX (CPU inference with `onnxruntime`) plus `classes.json`. Batch size is fixed to 1 (one uploaded photo per request). Preprocessing the backend must repeat: resize shorter side to 256, center crop 224, scale to 0-1, normalise with mean (0.485, 0.456, 0.406) and std (0.229, 0.224, 0.225).

In [ ]:
import onnxruntime as ort
dummy = torch.randn(1, 3, 224, 224, device=DEVICE)
torch.onnx.export(model, dummy, OUT / "food_classifier.onnx", input_names=["image"], output_names=["logits"],
                  opset_version=18)
json.dump({"classes": CLASSES}, open(OUT / "classes.json", "w"))

# new PyTorch writes weights to a side file (.onnx.data); fold them back so the model is ONE file
import onnx
onnx_path = str(OUT / "food_classifier.onnx")
onnx.save(onnx.load(onnx_path), onnx_path, save_as_external_data=False)
(OUT / "food_classifier.onnx.data").unlink(missing_ok=True)

# self-check: ONNX output matches PyTorch
sess = ort.InferenceSession(str(OUT / "food_classifier.onnx"), providers=["CPUExecutionProvider"])
with torch.no_grad(): ref = model(dummy).cpu().numpy()
got = sess.run(None, {"image": dummy.cpu().numpy()})[0]
assert got.shape == (1, len(CLASSES)) and np.allclose(ref, got, atol=1e-3), "ONNX export mismatch"
print("export OK ->", OUT)